# UV Decomposition via Gradient Descent

## Learning Objectives

1. **Define** UV (matrix factorization) as an optimization problem
2. **Implement** gradient descent to minimize reconstruction error
3. **Handle** missing values (for recommendation systems)
4. **Add** regularization to prevent overfitting

## Problem: Low-Rank Matrix Factorization

Given matrix $M \in \mathbb{R}^{m \times n}$, find $U \in \mathbb{R}^{m \times k}$, $V \in \mathbb{R}^{n \times k}$ minimizing:

$$\min_{U,V} \sum_{(i,j) \in \Omega} (M_{ij} - U_i \cdot V_j^\top)^2$$

where $\Omega$ = set of observed entries (can skip missing values).

**Why not just use SVD?** SVD requires all entries to be known. UV decomposition handles missing data by only minimizing over observed entries — essential for recommender systems.

## Gradient Descent

**Loss:**
$$L = \sum_{(i,j) \in \Omega} (M_{ij} - U_i \cdot V_j)^2$$

**Gradients:**
$$\frac{\partial L}{\partial U_{il}} = -2 \sum_{j:(i,j)\in\Omega} (M_{ij} - U_i \cdot V_j) V_{jl}$$
$$\frac{\partial L}{\partial V_{jl}} = -2 \sum_{i:(i,j)\in\Omega} (M_{ij} - U_i \cdot V_j) U_{il}$$

**SGD (Stochastic):** update on each observed entry $(i,j)$:
$$e_{ij} = M_{ij} - U_i \cdot V_j$$
$$U_i \leftarrow U_i + 2\eta \cdot e_{ij} \cdot V_j$$
$$V_j \leftarrow V_j + 2\eta \cdot e_{ij} \cdot U_i$$

In [1]:
import math, random

def dot(a, b): return sum(x*y for x,y in zip(a,b))

def uv_sgd(M_obs, m, n, k=3, eta=0.01, n_epochs=50, lam=0.01, seed=0):
    """UV decomposition via SGD with L2 regularization."""
    rng = random.Random(seed)
    # Initialize U and V with small random values
    U = [[rng.gauss(0, 0.1) for _ in range(k)] for _ in range(m)]
    V = [[rng.gauss(0, 0.1) for _ in range(k)] for _ in range(n)]
    obs = list(M_obs.items())
    losses = []
    for epoch in range(n_epochs):
        rng.shuffle(obs)
        total_loss = 0.0
        for (i,j), mij in obs:
            pred = dot(U[i], V[j])
            e = mij - pred
            # SGD update with L2 reg
            for l in range(k):
                U[i][l] += eta * (2*e*V[j][l] - lam*U[i][l])
                V[j][l] += eta * (2*e*U[i][l] - lam*V[j][l])
            total_loss += e**2
        losses.append(total_loss / len(obs))
    return U, V, losses

# Create a low-rank matrix with missing entries
rng = random.Random(1)
m, n, k_true = 10, 8, 2
# True U and V
U_true = [[rng.gauss(0,1) for _ in range(k_true)] for _ in range(m)]
V_true = [[rng.gauss(0,1) for _ in range(k_true)] for _ in range(n)]
M_full = [[dot(U_true[i], V_true[j]) + rng.gauss(0,0.1) for j in range(n)] for i in range(m)]

# Observe only 60% of entries
M_obs = {(i,j): M_full[i][j]
         for i in range(m) for j in range(n)
         if rng.random() < 0.6}
print(f"Total entries: {m*n}, Observed: {len(M_obs)} ({len(M_obs)*100//(m*n)}%)")

U_hat, V_hat, losses = uv_sgd(M_obs, m, n, k=2, eta=0.005, n_epochs=100)

print("""
Training loss (MSE on observed entries):""")
for ep in [0, 9, 24, 49, 99]:
    print(f"  Epoch {ep+1:>3}: {losses[ep]:.6f}")

Total entries: 80, Observed: 49 (61%)

Training loss (MSE on observed entries):
  Epoch   1: 0.521250
  Epoch  10: 0.515596
  Epoch  25: 0.499048
  Epoch  50: 0.415709
  Epoch 100: 0.197375


In [2]:
# Compute reconstruction error on ALL entries (including unobserved)
rmse_obs = math.sqrt(sum((M_full[i][j]-dot(U_hat[i],V_hat[j]))**2
                         for i,j in M_obs) / len(M_obs))
all_entries = [(i,j) for i in range(m) for j in range(n)]
unobs = [(i,j) for i,j in all_entries if (i,j) not in M_obs]
rmse_unobs = math.sqrt(sum((M_full[i][j]-dot(U_hat[i],V_hat[j]))**2
                           for i,j in unobs) / len(unobs))

print(f"RMSE on observed entries:   {rmse_obs:.4f}")
print(f"RMSE on unobserved entries: {rmse_unobs:.4f}")
print(f"""
Note: unobserved entries have slightly higher error (generalization).""")
print(f"This is the essence of collaborative filtering: predicting missing ratings.")


RMSE on observed entries:   0.4417
RMSE on unobserved entries: 1.2151

Note: unobserved entries have slightly higher error (generalization).
This is the essence of collaborative filtering: predicting missing ratings.


## Regularized UV

**L2 regularization** prevents $U$ and $V$ from having large norms:

$$L_{\text{reg}} = \sum_{(i,j)\in\Omega}(M_{ij} - U_i V_j^\top)^2 + \lambda(\|U\|_F^2 + \|V\|_F^2)$$

**SGD update:**
$$U_i \leftarrow U_i + \eta(2 e_{ij} V_j - \lambda U_i)$$
$$V_j \leftarrow V_j + \eta(2 e_{ij} U_i - \lambda V_j)$$

**Bias terms** (common in recommender systems): $\hat{M}_{ij} = \mu + b_i + c_j + U_i \cdot V_j$ where $\mu$ is global mean, $b_i$ is user bias, $c_j$ is item bias.

This variant was key to the Netflix Prize winning solution (Koren et al. 2009).